In [36]:
import requests
import pandas as pd
from datetime import datetime, timedelta

SERVICE_KEY = "f6e1ad2883e741c3cbdd0c9aa9be8aa4677dfeb08cdf1aa1a2a243d22183e89f"
URL = "https://apis.data.go.kr/B551015/API186_1/SeoulRace_1"

def safe_get_items(data):
    try:
        body = data.get("response", {}).get("body", {})
        items = body.get("items", {})

        if not items:
            return []

        item = items.get("item", None)

        if item is None or item == "":
            return []

        if isinstance(item, dict):
            return [item]

        elif isinstance(item, list):
            return item

        else:
            return []

    except:
        return []

# ----------------------
# API 호출 (page 포함)
# ----------------------
def get_data(start_date, end_date, page):
    params = {
        "ServiceKey": SERVICE_KEY,
        "pageNo": str(page),
        "numOfRows": "10",   # ⭐ 안정값
        "rc_date_fr": start_date,
        "rc_date_to": end_date,
        "_type": "json"
    }

    response = requests.get(URL, params=params)

    if response.status_code != 200:
        print("요청 실패:", response.status_code)
        return [], 0

    try:
        data = response.json()
        total = data.get("response", {}).get("body", {}).get("totalCount", 0)
    except:
        print("JSON 실패")
        return [], 0

    items = safe_get_items(data)
    return items, total

# ----------------------
# 🔄 날짜 + 페이지 병행
# ----------------------
start = datetime(2021, 10, 1)
end = datetime(2021, 12, 31)

all_data = []
current = start

while current <= end:
    next_date = current + timedelta(days=7)   # ⭐ 핵심 수정

    print(f"{current.date()} ~ {next_date.date()}")

    # 첫 페이지
    items, total = get_data(
        current.strftime("%Y%m%d"),
        next_date.strftime("%Y%m%d"),
        1
    )

    all_data.extend(items)

    page_size = 500
    total_pages = (total // page_size) + 1

    for page in range(2, total_pages + 1):
        items, _ = get_data(
            current.strftime("%Y%m%d"),
            next_date.strftime("%Y%m%d"),
            page
        )
        all_data.extend(items)

    current = next_date

# ----------------------
# 결과
# ----------------------
df = pd.DataFrame(all_data)

print("총 데이터 개수:", len(df))
print(df.head())

df.to_csv("data_4.csv", index=False)

2021-10-01 ~ 2021-10-08
2021-10-08 ~ 2021-10-15
2021-10-15 ~ 2021-10-22
2021-10-22 ~ 2021-10-29
2021-10-29 ~ 2021-11-05
2021-11-05 ~ 2021-11-12
2021-11-12 ~ 2021-11-19
2021-11-19 ~ 2021-11-26
2021-11-26 ~ 2021-12-03
2021-12-03 ~ 2021-12-10
2021-12-10 ~ 2021-12-17
2021-12-17 ~ 2021-12-24
2021-12-24 ~ 2021-12-31
2021-12-31 ~ 2022-01-07
총 데이터 개수: 130
    chaksun  diffTot  divide hrName     hrno jkName    jkNo meet noracefl  \
0         0    17.25       0  야무진왕초  0044009    조한별  080515   서울       정상   
1  13200000     1.25       0   와우와우  0045162    송재철  080513   서울       정상   
2         0    23.50       0  새내파워풀  0044100    함완식  080342   서울       정상   
3   2400000    10.75       0   트리플런  0043798    장추열  080476   서울       정상   
4   3000000     4.75       0   라온더펄  0044204     이혁  080486   서울       정상   

     prow  ... rcRank rcSex rcSpcba  rcSpcbu  rcTime rcVtdusu  rundayth  \
0  111006  ...     06    오픈     0.0        1    63.4       12        60   
1  110034  ...     06    오픈     0.0  